In [48]:
library(data.table)

base_dir <- "/home/ds/buckets/b1/exp/WFA912"


In [49]:
# ============================================================
# DETECTAR AUTOMÁTICAMENTE LAS SEMILLAS DISPONIBLES
# ============================================================

archivos <- list.files(
  path = base_dir,
  pattern = "^prediccion\\.txt[0-9]+$",
  full.names = TRUE
)

semillas <- as.integer(
  sub("^prediccion\\.txt", "", basename(archivos))
)

semillas <- sort(semillas)

cat("Semillas encontradas:", length(semillas), "\n")
print(semillas)

Semillas encontradas: 100 
  [1] 100003 100019 103801 118369 125003 136733 150011 155087 173473 175013
 [11] 191837 200017 210209 214651 225023 228577 246937 250027 265313 275027
 [21] 283669 291173 300043 302053 317537 320401 325043 325537 338777 350087
 [31] 357139 375091 375509 393871 400093 400123 412249 425101 426389 430603
 [41] 437459 448969 450101 467353 475103 485717 500029 500107 504073 522449
 [51] 525127 540809 548621 550127 559183 573749 575129 577547 595927 600167
 [61] 614291 625169 632647 650179 651019 659947 669391 675179 684769 687749
 [71] 689167 700199 706117 712687 724487 742891 761227 763181 779591 797957
 [81] 811501 816329 824831 834703 853057 856621 871439 887789 889783 908153
 [91] 918763 926533 931241 944897 947317 963253 967559 981637 981799 999983


In [50]:
# ============================================================
# LEER TODAS LAS PREDICCIONES
# ============================================================

predicciones <- lapply(
  semillas,
  function(semilla) {

    archivo <- paste0(
      base_dir,
      "/prediccion.txt",
      semilla
    )

    tb <- fread(archivo)

    tb[, semilla := semilla]

    return(tb)
  }
)

names(predicciones) <- semillas

In [51]:
# ============================================================
# MATRIZ DE PROBABILIDADES
# ============================================================

probabilidades <- sapply(
  predicciones,
  function(tb) tb$prob
)

colnames(probabilidades) <- semillas

ensemble <- predicciones[[1]][
  ,
  .(
    numero_de_cliente,
    clase_ternaria
  )
]

ensemble[
  ,
  prob_ensemble := rowMeans(probabilidades)
]

In [52]:
cat("\n========================================\n")
cat("ENSEMBLE\n")
cat("========================================\n")
cat("Cantidad de modelos:", length(semillas), "\n")
cat("Cantidad de clientes:", nrow(ensemble), "\n")
cat("========================================\n")

head(ensemble)


ENSEMBLE
Cantidad de modelos: 100 
Cantidad de clientes: 33080 


numero_de_cliente,clase_ternaria,prob_ensemble
<int>,<lgl>,<dbl>
11675570,NA,0.002342383
11675731,NA,0.001120676
11677630,NA,0.007268527
11678491,NA,0.001626552
11678621,NA,0.001632546
11679331,NA,0.002442438


In [53]:
library(data.table)

# ============================================================
# CONFIGURACIÓN KAGGLE
# ============================================================

exp <- "912"

base_dir <- paste0(
  "/home/ds/buckets/b1/exp/WFA",
  exp
)

kaggle_dir <- file.path(
  base_dir,
  "kaggle"
)

competencia <- "data-mining-junior-2026-b"


# ============================================================
# DETECTAR CANTIDAD DE SEMILLAS
# ============================================================

archivos <- list.files(
  path = base_dir,
  pattern = "^prediccion\\.txt[0-9]+$",
  full.names = TRUE
)

if (length(archivos) == 0) {
  stop("No se encontraron archivos prediccion.txt<semilla>")
}

semillas <- sub(
  "^prediccion\\.txt",
  "",
  basename(archivos)
)

nsemillas <- length(semillas)

cat(
  "Cantidad de modelos del ensemble:",
  nsemillas,
  "\n"
)

cat(
  "Semillas:",
  paste(semillas, collapse = ", "),
  "\n"
)


# ============================================================
# VERIFICAR ENSEMBLE EN MEMORIA
# ============================================================

if (!exists("ensemble")) {
  stop(
    "El objeto 'ensemble' no existe en memoria."
  )
}

cat(
  "Ensemble encontrado en memoria.\n"
)

cat(
  "Clientes:",
  nrow(ensemble),
  "\n"
)


# ============================================================
# COPIA PARA KAGGLE
# ============================================================

tb_kaggle <- copy(ensemble)

# Ordenar por probabilidad del ensemble
setorder(
  tb_kaggle,
  -prob_ensemble
)


# ============================================================
# CORTES
# ============================================================

#grueso
#cortes <- seq(
#  1200,
#  2100,
#  by = 100
#)

#fino
#cortes <- seq(
#  1710,
#  1890,
#  by = 10
#)

cortes <- seq(
  1690,
  1720,
  by = 2
)

cortes <- cortes[cortes != 1800]

cat(
  "Cortes:",
  paste(cortes, collapse = ", "),
  "\n"
)


# ============================================================
# CREAR DIRECTORIO KAGGLE
# ============================================================

dir.create(
  kaggle_dir,
  showWarnings = FALSE
)


# ============================================================
# GENERAR Y ENVIAR
# ============================================================

for (envios in cortes) {

  archivo_kaggle <- file.path(
    kaggle_dir,
    paste0(
      "KA",
      exp,
      "_ensemble_",
      nsemillas,
      "_",
      envios,
      ".csv"
    )
  )


  # ----------------------------------------------------------
  # Crear clasificación binaria
  # ----------------------------------------------------------

  tb_kaggle[
    ,
    Predicted := FALSE
  ]

  tb_kaggle[
    1:envios,
    Predicted := TRUE
  ]


  # ----------------------------------------------------------
  # Guardar CSV
  # ----------------------------------------------------------

  fwrite(
    tb_kaggle[
      ,
      .(
        numero_de_cliente,
        Predicted
      )
    ],
    file = archivo_kaggle,
    sep = ","
  )


  cat(
    "\n========================================\n"
  )

  cat(
    "Enviando Ensemble:",
    nsemillas,
    "semillas\n"
  )

  cat(
    "Envíos:",
    envios,
    "\n"
  )

  cat(
    "Archivo:",
    basename(archivo_kaggle),
    "\n"
  )

  cat(
    "========================================\n"
  )

  flush.console()


  # ----------------------------------------------------------
  # Submission
  # ----------------------------------------------------------

  comando <- paste(
    "kaggle competitions submit",
    "-c", competencia,
    "-f", shQuote(archivo_kaggle),
    "-m", shQuote(
      paste0(
        "WFA", exp,
        " Ensemble ",
        nsemillas,
        " semillas",
        " envios=", envios
      )
    )
  )

  salida <- system(
    comando,
    intern = TRUE
  )

  cat(
    paste(
      salida,
      collapse = "\n"
    ),
    "\n"
  )


  # ----------------------------------------------------------
  # Esperar
  # ----------------------------------------------------------

  if (envios != max(cortes)) {

    cat(
      "Esperando 10 segundos...\n"
    )

    flush.console()

    Sys.sleep(10)
  }
}


cat(
  "\n========================================\n"
)

cat(
  "TODOS LOS ENVIOS DEL ENSEMBLE FINALIZADOS\n"
)

cat(
  "Modelos:",
  nsemillas,
  "\n"
)

cat(
  "Cantidad de envíos:",
  length(cortes),
  "\n"
)

cat(
  "========================================\n"
)

Cantidad de modelos del ensemble: 100 
Semillas: 100003, 100019, 103801, 118369, 125003, 136733, 150011, 155087, 173473, 175013, 191837, 200017, 210209, 214651, 225023, 228577, 246937, 250027, 265313, 275027, 283669, 291173, 300043, 302053, 317537, 320401, 325043, 325537, 338777, 350087, 357139, 375091, 375509, 393871, 400093, 400123, 412249, 425101, 426389, 430603, 437459, 448969, 450101, 467353, 475103, 485717, 500029, 500107, 504073, 522449, 525127, 540809, 548621, 550127, 559183, 573749, 575129, 577547, 595927, 600167, 614291, 625169, 632647, 650179, 651019, 659947, 669391, 675179, 684769, 687749, 689167, 700199, 706117, 712687, 724487, 742891, 761227, 763181, 779591, 797957, 811501, 816329, 824831, 834703, 853057, 856621, 871439, 887789, 889783, 908153, 918763, 926533, 931241, 944897, 947317, 963253, 967559, 981637, 981799, 999983 
Ensemble encontrado en memoria.
Clientes: 33080 
Cortes: 1690, 1692, 1694, 1696, 1698, 1700, 1702, 1704, 1706, 1708, 1710, 1712, 1714, 1716, 1718, 1720

In [54]:
cat("Listo")

Listo